In [3]:
# ============================================================
# EXPERIMENT 18
# RESNET18 BASELINE + FPN
#
# SELF-CONTAINED NOTEBOOK
# ============================================================

import os
import sys
import time
import json
import pickle
import random
import hashlib
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import boto3
import numpy as np
import pandas as pd

from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import (
    resnet18,
    ResNet18_Weights
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# AWS
# ============================================================

BUCKET_NAME = (
    "amal-stitchingnet-data-903473565738-ap-southeast-2-an"
)

REGION = "ap-southeast-2"

DATA_PREFIX = "data/"

# ============================================================
# PROJECT DIRECTORIES
# ============================================================

PROJECT_ROOT = Path.cwd()

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "training_cache"
    / "original_rgb"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "models"
    / "exp18_resnet18_fpn"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "models"
    / "exp18_resnet18_fpn"
)

for directory in [
    CACHE_DIR,
    OUTPUT_DIR,
    REPORT_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ============================================================
# MODEL CONFIGURATION
# ============================================================

NUM_CLASSES = 11

IMAGE_SIZE = 224

BATCH_SIZE = 64

MAX_EPOCHS = 20
MIN_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 4

BACKBONE_LR = 1e-4
NEW_LAYERS_LR = 5e-4

WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.10

NUM_WORKERS = min(
    4,
    os.cpu_count() or 1
)

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = DEVICE.type == "cuda"

print("=" * 70)
print("EXPERIMENT 18 CONFIGURATION")
print("=" * 70)

print("Project root:", PROJECT_ROOT)
print("Cache:", CACHE_DIR)
print("Models:", OUTPUT_DIR)
print("Reports:", REPORT_DIR)

print("\nDevice:", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0)
            .total_memory
            / (1024 ** 3),
            2
        ),
        "GB"
    )

print("\nBatch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)
print("Early stopping:", EARLY_STOPPING_PATIENCE)
print("Workers:", NUM_WORKERS)

print("\nConfiguration ready.")

EXPERIMENT 18 CONFIGURATION
Project root: /home/ec2-user/SageMaker/stitching-defect-detection/Notebooks
Cache: /home/ec2-user/SageMaker/stitching-defect-detection/Notebooks/data/training_cache/original_rgb
Models: /home/ec2-user/SageMaker/stitching-defect-detection/Notebooks/models/exp18_resnet18_fpn
Reports: /home/ec2-user/SageMaker/stitching-defect-detection/Notebooks/reports/models/exp18_resnet18_fpn

Device: cpu

Batch size: 64
Maximum epochs: 20
Early stopping: 4
Workers: 2

Configuration ready.


In [4]:
# ============================================================
# DATASET MANIFEST DISCOVERY
# ============================================================

s3 = boto3.client(
    "s3",
    region_name=REGION
)

s3.head_bucket(
    Bucket=BUCKET_NAME
)

print("S3 connection: OK")

# ------------------------------------------------------------
# Search project for candidate split manifests
# ------------------------------------------------------------

candidate_files = []

for csv_file in PROJECT_ROOT.rglob("*.csv"):

    try:
        sample = pd.read_csv(
            csv_file,
            nrows=5
        )

        columns = set(
            sample.columns
        )

        required = {
            "s3_key",
            "folder_1",
            "folder_2",
            "split"
        }

        if required.issubset(columns):

            candidate_files.append(
                csv_file
            )

    except Exception:
        continue

print("\nCandidate split manifests:")

for file in candidate_files:

    print(
        " -",
        file
    )

assert candidate_files, (
    "No valid dataset split manifest found."
)

# ------------------------------------------------------------
# Prefer the most recently modified valid manifest.
# This should select the corrected split if it was saved
# after the original split.
# ------------------------------------------------------------

candidate_files = sorted(
    candidate_files,
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

MANIFEST_PATH = candidate_files[0]

print(
    "\nSelected manifest:",
    MANIFEST_PATH
)

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

split_df = pd.read_csv(
    MANIFEST_PATH
)

print(
    "\nRows:",
    len(split_df)
)

print(
    "Columns:",
    split_df.columns.tolist()
)

display(
    split_df.head()
)

S3 connection: OK

Candidate split manifests:
 - /home/ec2-user/SageMaker/stitching-defect-detection/Notebooks/reports/eda/exp16_duplicate_detection/dataset_split_manifest_corrected.csv
 - /home/ec2-user/SageMaker/stitching-defect-detection/Notebooks/reports/eda/exp16_duplicate_detection/sha256_manifest.csv
 - /home/ec2-user/SageMaker/stitching-defect-detection/Notebooks/reports/eda/exp16_duplicate_detection/exact_duplicate_images.csv
 - /home/ec2-user/SageMaker/stitching-defect-detection/Notebooks/reports/eda/exp15_dataset_splits/dataset_split_manifest.csv
 - /home/ec2-user/SageMaker/stitching-defect-detection/Notebooks/reports/eda/exp17_preprocessing/preprocessed_manifest.csv
 - /home/ec2-user/SageMaker/stitching-defect-detection/Notebooks/reports/eda/exp17_preprocessing/preprocessing_progress.csv

Selected manifest: /home/ec2-user/SageMaker/stitching-defect-detection/Notebooks/reports/eda/exp17_preprocessing/preprocessed_manifest.csv

Rows: 14565
Columns: ['sample_id', 'file_name', 

,sample_id,file_name,s3_key,lab_clahe_s3_key,bilateral_clahe_s3_key,folder_1,folder_2,split,original_size_bytes,lab_size_bytes,bilateral_size_bytes,height,width,status
0,728aa2c160203dd5,G_10_027.jpg,data/G. Chiffon-Poly/10. Overlapped stitch/G_1...,processed-data/exp17_multiview/lab_clahe/G. Ch...,processed-data/exp17_multiview/bilateral_clahe...,G. Chiffon-Poly,10. Overlapped stitch,train,70789,35325,12179,224,224,success
1,b42496a424e23121,H_04_106.jpg,data/H. Nylon-Core/4. Crooked seam/H_04_106.jpg,processed-data/exp17_multiview/lab_clahe/H. Ny...,processed-data/exp17_multiview/bilateral_clahe...,H. Nylon-Core,4. Crooked seam,train,61010,14559,14991,224,224,success
2,6f691ecc66dc1ffd,F_02_083.jpg,data/F. Satin-Core/2. Broken stitch/F_02_083.jpg,processed-data/exp17_multiview/lab_clahe/F. Sa...,processed-data/exp17_multiview/bilateral_clahe...,F. Satin-Core,2. Broken stitch,train,57803,24358,11632,224,224,success
3,eb4bcbcb9a34e528,E_08_011.jpg,data/E. Polyester-Poly/8. Needle mark/E_08_011...,processed-data/exp17_multiview/lab_clahe/E. Po...,processed-data/exp17_multiview/bilateral_clahe...,E. Polyester-Poly,8. Needle mark,train,59211,21573,15958,224,224,success
4,8625ca7cb74cd270,I_03_029.jpg,data/I. Jacquard-Poly/3. Pinched fabric/I_03_0...,processed-data/exp17_multiview/lab_clahe/I. Ja...,processed-data/exp17_multiview/bilateral_clahe...,I. Jacquard-Poly,3. Pinched fabric,train,79640,37769,29454,224,224,success


In [5]:
# ============================================================
# SPLIT VALIDATION
# ============================================================

required_columns = {
    "s3_key",
    "folder_1",
    "folder_2",
    "split"
}

missing = (
    required_columns
    - set(split_df.columns)
)

assert not missing, (
    f"Missing columns: {missing}"
)

# ------------------------------------------------------------
# Basic completeness
# ------------------------------------------------------------

assert len(split_df) == 14565, (
    f"Expected 14565 images, "
    f"found {len(split_df)}"
)

assert (
    split_df["s3_key"].notna().all()
)

assert (
    split_df["folder_1"].notna().all()
)

assert (
    split_df["folder_2"].notna().all()
)

# ------------------------------------------------------------
# No duplicate S3 keys
# ------------------------------------------------------------

duplicate_keys = (
    split_df["s3_key"]
    .duplicated()
    .sum()
)

assert duplicate_keys == 0, (
    f"Duplicate S3 keys found: {duplicate_keys}"
)

# ------------------------------------------------------------
# Class IDs
# ------------------------------------------------------------

split_df["class_id"] = (
    split_df["folder_2"]
    .str.extract(r"^(\d+)")
    .astype(int)
)

assert (
    sorted(
        split_df["class_id"].unique()
    )
    == list(range(NUM_CLASSES))
)

# ------------------------------------------------------------
# Split names
# ------------------------------------------------------------

allowed_splits = {
    "train",
    "validation",
    "test"
}

assert (
    set(split_df["split"].unique())
    == allowed_splits
)

# ------------------------------------------------------------
# Class names
# ------------------------------------------------------------

class_names = (
    split_df[
        ["class_id", "folder_2"]
    ]
    .drop_duplicates()
    .sort_values("class_id")
    .set_index("class_id")["folder_2"]
    .to_dict()
)

# ------------------------------------------------------------
# Split distribution
# ------------------------------------------------------------

print("=" * 70)
print("SPLIT VALIDATION")
print("=" * 70)

print("\nSplit distribution:")

display(
    split_df["split"]
    .value_counts()
    .sort_index()
    .rename("image_count")
    .to_frame()
)

print("\nClass distribution:")

display(
    pd.crosstab(
        split_df["folder_2"],
        split_df["split"]
    )
)

print("\nFabric distribution:")

display(
    pd.crosstab(
        split_df["folder_1"],
        split_df["split"]
    )
)

print("\nClass mapping:")

for class_id, name in class_names.items():

    print(
        f"{class_id:2d} -> {name}"
    )

print("\nSplit validation PASSED.")

SPLIT VALIDATION

Split distribution:


,image_count
split,
test,2208
train,10154
validation,2203



Class distribution:


split,test,train,validation
folder_2,,,
0. Normal,376,1690,370
1. Skipped stitch,186,867,186
10. Overlapped stitch,180,843,180
2. Broken stitch,183,835,183
3. Pinched fabric,180,834,178
4. Crooked seam,187,856,187
5. Thread sagging,178,819,176
6. Puckering,187,867,186
7. Stain and damage,176,806,180



Fabric distribution:


split,test,train,validation
folder_1,,,
A. Cotton-Poly,210,974,210
B. Linen-Poly,201,931,201
C. Denim-Poly,192,892,194
D. Velveteen-Poly,202,868,201
E. Polyester-Poly,206,940,205
F. Satin-Core,193,912,193
G. Chiffon-Poly,192,887,191
H. Nylon-Core,208,963,204
I. Jacquard-Poly,195,881,192



Class mapping:
 0 -> 0. Normal
 1 -> 1. Skipped stitch
 2 -> 2. Broken stitch
 3 -> 3. Pinched fabric
 4 -> 4. Crooked seam
 5 -> 5. Thread sagging
 6 -> 6. Puckering
 7 -> 7. Stain and damage
 8 -> 8. Needle mark
 9 -> 9. Bobbin thread pulling up
10 -> 10. Overlapped stitch

Split validation PASSED.


In [6]:
# ============================================================
# LOCAL CACHE PATHS
# ============================================================

def local_path_from_s3_key(s3_key):

    relative = s3_key

    if relative.startswith(
        DATA_PREFIX
    ):
        relative = relative[
            len(DATA_PREFIX):
        ]

    return CACHE_DIR / relative


split_df["local_path"] = (
    split_df["s3_key"]
    .apply(local_path_from_s3_key)
)

print(
    "Example local paths:"
)

display(
    split_df[
        [
            "s3_key",
            "local_path",
            "split",
            "folder_2"
        ]
    ].head()
)

# ------------------------------------------------------------
# Estimate source storage
# ------------------------------------------------------------

if "file_size_kb" in split_df.columns:

    total_gb = (
        split_df["file_size_kb"].sum()
        / 1024
        / 1024
    )

    print(
        "\nEstimated source size:",
        round(total_gb, 3),
        "GB"
    )

Example local paths:


,s3_key,local_path,split,folder_2
0,data/G. Chiffon-Poly/10. Overlapped stitch/G_1...,/home/ec2-user/SageMaker/stitching-defect-dete...,train,10. Overlapped stitch
1,data/H. Nylon-Core/4. Crooked seam/H_04_106.jpg,/home/ec2-user/SageMaker/stitching-defect-dete...,train,4. Crooked seam
2,data/F. Satin-Core/2. Broken stitch/F_02_083.jpg,/home/ec2-user/SageMaker/stitching-defect-dete...,train,2. Broken stitch
3,data/E. Polyester-Poly/8. Needle mark/E_08_011...,/home/ec2-user/SageMaker/stitching-defect-dete...,train,8. Needle mark
4,data/I. Jacquard-Poly/3. Pinched fabric/I_03_0...,/home/ec2-user/SageMaker/stitching-defect-dete...,train,3. Pinched fabric


In [7]:
# ============================================================
# DOWNLOAD ORIGINAL RGB DATASET
# ============================================================

def download_one(record):

    s3_key = record.s3_key
    local_path = Path(record.local_path)

    local_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Skip if already downloaded.
    if (
        local_path.exists()
        and local_path.stat().st_size > 0
    ):
        return "existing"

    s3.download_file(
        BUCKET_NAME,
        s3_key,
        str(local_path)
    )

    return "downloaded"


records = list(
    split_df[
        [
            "s3_key",
            "local_path"
        ]
    ].itertuples(
        index=False
    )
)

start_time = time.time()

downloaded = 0
existing = 0
failed = []

print(
    f"Preparing {len(records)} images..."
)

# 8 parallel downloads.
# Enough to make S3 transfer fast without
# creating excessive local contention.

with ThreadPoolExecutor(
    max_workers=8
) as executor:

    future_map = {
        executor.submit(
            download_one,
            record
        ): record
        for record in records
    }

    for i, future in enumerate(
        as_completed(future_map),
        start=1
    ):

        record = future_map[future]

        try:

            result = future.result()

            if result == "downloaded":
                downloaded += 1

            else:
                existing += 1

        except Exception as exc:

            failed.append(
                {
                    "s3_key": record.s3_key,
                    "error": repr(exc)
                }
            )

        if (
            i % 500 == 0
            or i == len(records)
        ):

            print(
                f"{i}/{len(records)}"
            )

elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("DOWNLOAD COMPLETE")
print("=" * 70)

print(
    "Downloaded:",
    downloaded
)

print(
    "Already cached:",
    existing
)

print(
    "Failed:",
    len(failed)
)

print(
    "Time:",
    round(elapsed / 60, 2),
    "minutes"
)

assert not failed, (
    f"{len(failed)} downloads failed. "
    f"Do not start training."
)

Preparing 14565 images...
500/14565
1000/14565
1500/14565
2000/14565
2500/14565
3000/14565
3500/14565
4000/14565
4500/14565
5000/14565
5500/14565
6000/14565
6500/14565
7000/14565
7500/14565
8000/14565
8500/14565
9000/14565
9500/14565
10000/14565
10500/14565
11000/14565
11500/14565
12000/14565
12500/14565
13000/14565
13500/14565
14000/14565
14500/14565
14565/14565

DOWNLOAD COMPLETE
Downloaded: 14565
Already cached: 0
Failed: 0
Time: 3.05 minutes


In [8]:
# ============================================================
# LOCAL IMAGE QUALITY VERIFICATION
# ============================================================

bad_images = []

for i, row in enumerate(
    split_df.itertuples(),
    start=1
):

    path = Path(row.local_path)

    try:

        if not path.exists():
            raise FileNotFoundError(
                "File missing"
            )

        if path.stat().st_size == 0:
            raise ValueError(
                "Empty file"
            )

        with Image.open(path) as img:

            img.verify()

        # Re-open to verify decoding.
        with Image.open(path) as img:

            img = img.convert("RGB")

            if img.size != (
                IMAGE_SIZE,
                IMAGE_SIZE
            ):

                raise ValueError(
                    f"Unexpected size: "
                    f"{img.size}"
                )

    except Exception as exc:

        bad_images.append(
            {
                "s3_key": row.s3_key,
                "local_path": str(path),
                "error": repr(exc)
            }
        )

    if (
        i % 1000 == 0
        or i == len(split_df)
    ):

        print(
            f"Verified {i}/{len(split_df)}"
        )

print(
    "\nBad images:",
    len(bad_images)
)

assert not bad_images, (
    "Image verification failed."
)

print(
    "ALL 14,565 LOCAL IMAGES VERIFIED."
)

Verified 1000/14565
Verified 2000/14565
Verified 3000/14565
Verified 4000/14565
Verified 5000/14565
Verified 6000/14565
Verified 7000/14565
Verified 8000/14565
Verified 9000/14565
Verified 10000/14565
Verified 11000/14565
Verified 12000/14565
Verified 13000/14565
Verified 14000/14565
Verified 14565/14565

Bad images: 0
ALL 14,565 LOCAL IMAGES VERIFIED.


In [ ]:
# ============================================================
# PYTORCH DATASET
# ============================================================

class StitchingDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        image = Image.open(
            row["local_path"]
        ).convert("RGB")

        if self.transform is not None:

            image = self.transform(
                image
            )

        label = int(
            row["class_id"]
        )

        return (
            image,
            label
        )


# ============================================================
# TRANSFORMS
# ============================================================

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]

train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    # Horizontal direction can change,
    # but the horizontal seam structure remains.
    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    # Small geometric variation.
    # We intentionally avoid aggressive crops.
    transforms.RandomAffine(
        degrees=3,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
        interpolation=
            transforms.InterpolationMode.BILINEAR
    ),

    # Small appearance variation.
    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08,
        saturation=0.05,
        hue=0.0
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


eval_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ============================================================
# SPLIT DATAFRAMES
# ============================================================

train_df = (
    split_df[
        split_df["split"] == "train"
    ]
    .reset_index(drop=True)
)

validation_df = (
    split_df[
        split_df["split"] == "validation"
    ]
    .reset_index(drop=True)
)

test_df = (
    split_df[
        split_df["split"] == "test"
    ]
    .reset_index(drop=True)
)


# ============================================================
# DATASETS
# ============================================================

train_dataset = StitchingDataset(
    train_df,
    transform=train_transform
)

validation_dataset = StitchingDataset(
    validation_df,
    transform=eval_transform
)

test_dataset = StitchingDataset(
    test_df,
    transform=eval_transform
)


# ============================================================
# DATALOADERS
# ============================================================

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
}

if NUM_WORKERS > 0:

    loader_kwargs[
        "persistent_workers"
    ] = True


train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    drop_last=True,
    **loader_kwargs
)

validation_loader = DataLoader(
    validation_dataset,
    shuffle=False,
    **loader_kwargs
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **loader_kwargs
)


print("=" * 70)
print("DATALOADERS READY")
print("=" * 70)

print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(validation_dataset)
)

print(
    "Test:",
    len(test_dataset)
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

In [ ]:
# ============================================================
# DATALOADER SANITY CHECK
# ============================================================

images, labels = next(
    iter(train_loader)
)

print(
    "Image batch shape:",
    images.shape
)

print(
    "Label batch shape:",
    labels.shape
)

print(
    "Image dtype:",
    images.dtype
)

print(
    "Label dtype:",
    labels.dtype
)

print(
    "Image device:",
    images.device
)

print(
    "Label range:",
    labels.min().item(),
    "to",
    labels.max().item()
)

assert images.shape[1:] == (
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
)

assert labels.ndim == 1

assert (
    labels.min().item() >= 0
)

assert (
    labels.max().item()
    < NUM_CLASSES
)

print(
    "\nDATALOADER SANITY CHECK PASSED."
)

In [ ]:
# ============================================================
# BASELINE RESNET18
# ============================================================

def build_resnet18():

    weights = (
        ResNet18_Weights.IMAGENET1K_V1
    )

    model = resnet18(
        weights=weights
    )

    # Replace ImageNet's 1000-class
    # classifier with our 11-class classifier.

    input_features = (
        model.fc.in_features
    )

    model.fc = nn.Linear(
        input_features,
        NUM_CLASSES
    )

    return model


baseline_model = build_resnet18()

baseline_model = (
    baseline_model.to(DEVICE)
)


def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
    )


print(
    "ResNet18 parameters:",
    f"{count_parameters(baseline_model):,}"
)

print(
    "Output classes:",
    NUM_CLASSES
)

print(
    "\nBaseline model ready."
)

In [ ]:
# ============================================================
# TRAINING ENGINE
# ============================================================

def make_optimizer(
    model,
    new_layer_names
):

    backbone_params = []
    new_params = []

    for name, parameter in model.named_parameters():

        if not parameter.requires_grad:
            continue

        is_new = any(
            name.startswith(prefix)
            for prefix in new_layer_names
        )

        if is_new:
            new_params.append(parameter)
        else:
            backbone_params.append(parameter)

    parameter_groups = [
        {
            "params": backbone_params,
            "lr": BACKBONE_LR
        },
        {
            "params": new_params,
            "lr": NEW_LAYERS_LR
        }
    ]

    optimizer = optim.AdamW(
        parameter_groups,
        weight_decay=WEIGHT_DECAY
    )

    return optimizer


def calculate_metrics(
    targets,
    predictions
):

    accuracy = accuracy_score(
        targets,
        predictions
    )

    macro_f1 = f1_score(
        targets,
        predictions,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        targets,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }


def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    scheduler=None
):

    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_items = 0

    all_targets = []
    all_predictions = []

    scaler = None

    if training and AMP_ENABLED:

        scaler = torch.amp.GradScaler(
            "cuda"
        )

    for images, labels in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if training:

            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(
            training
        ):

            if AMP_ENABLED:

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):

                    outputs = model(
                        images
                    )

                    loss = criterion(
                        outputs,
                        labels
                    )

            else:

                outputs = model(
                    images
                )

                loss = criterion(
                    outputs,
                    labels
                )

            if training:

                if AMP_ENABLED:

                    scaler.scale(
                        loss
                    ).backward()

                    scaler.unscale_(
                        optimizer
                    )

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=1.0
                    )

                    scaler.step(
                        optimizer
                    )

                    scaler.update()

                else:

                    loss.backward()

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=1.0
                    )

                    optimizer.step()

                if scheduler is not None:

                    scheduler.step()

        batch_size = labels.size(0)

        total_loss += (
            loss.item()
            * batch_size
        )

        total_items += batch_size

        predictions = (
            outputs
            .argmax(dim=1)
            .detach()
            .cpu()
            .numpy()
        )

        targets = (
            labels
            .detach()
            .cpu()
            .numpy()
        )

        all_predictions.extend(
            predictions.tolist()
        )

        all_targets.extend(
            targets.tolist()
        )

    metrics = calculate_metrics(
        all_targets,
        all_predictions
    )

    metrics["loss"] = (
        total_loss
        / total_items
    )

    return metrics


def save_checkpoint(
    model,
    model_name,
    epoch,
    validation_metrics,
    training_history
):

    checkpoint = {

        "model_name": model_name,

        "epoch": epoch,

        "model_state_dict":
            model.state_dict(),

        "num_classes":
            NUM_CLASSES,

        "class_names":
            class_names,

        "image_size":
            IMAGE_SIZE,

        "validation_metrics":
            validation_metrics,

        "training_history":
            training_history,

        "seed":
            SEED
    }

    # PyTorch-native checkpoint
    pth_path = (
        OUTPUT_DIR
        / f"{model_name}_best.pth"
    )

    torch.save(
        checkpoint,
        pth_path
    )

    # Pickle package requested for the project.
    pkl_path = (
        OUTPUT_DIR
        / f"{model_name}_best.pkl"
    )

    with open(
        pkl_path,
        "wb"
    ) as file:

        pickle.dump(
            checkpoint,
            file
        )

    return pth_path, pkl_path


def train_model(
    model,
    model_name,
    new_layer_names
):

    print("\n" + "=" * 70)
    print(
        f"TRAINING: {model_name}"
    )
    print("=" * 70)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING
    )

    optimizer = make_optimizer(
        model,
        new_layer_names
    )

    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[
            BACKBONE_LR,
            NEW_LAYERS_LR
        ],
        epochs=MAX_EPOCHS,
        steps_per_epoch=len(
            train_loader
        ),
        pct_start=0.30,
        anneal_strategy="cos"
    )

    history = []

    best_macro_f1 = -np.inf
    best_epoch = None
    patience_counter = 0

    training_start = time.time()

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        epoch_start = time.time()

        train_metrics = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            scheduler=scheduler
        )

        validation_metrics = run_epoch(
            model,
            validation_loader,
            criterion
        )

        epoch_time = (
            time.time()
            - epoch_start
        )

        record = {

            "epoch": epoch,

            "train_loss":
                train_metrics["loss"],

            "train_accuracy":
                train_metrics["accuracy"],

            "train_macro_f1":
                train_metrics["macro_f1"],

            "val_loss":
                validation_metrics["loss"],

            "val_accuracy":
                validation_metrics["accuracy"],

            "val_macro_f1":
                validation_metrics["macro_f1"],

            "val_weighted_f1":
                validation_metrics[
                    "weighted_f1"
                ],

            "epoch_seconds":
                epoch_time
        }

        history.append(record)

        print(
            f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"Train F1={train_metrics['macro_f1']:.4f} | "
            f"Val F1={validation_metrics['macro_f1']:.4f} | "
            f"Val Acc={validation_metrics['accuracy']:.4f} | "
            f"{epoch_time:.1f}s"
        )

        # ----------------------------------------------------
        # Save best model using validation macro F1.
        # ----------------------------------------------------

        if (
            validation_metrics["macro_f1"]
            > best_macro_f1
        ):

            best_macro_f1 = (
                validation_metrics[
                    "macro_f1"
                ]
            )

            best_epoch = epoch

            patience_counter = 0

            save_checkpoint(
                model,
                model_name,
                epoch,
                validation_metrics,
                history
            )

            print(
                "  -> BEST MODEL SAVED"
            )

        else:

            patience_counter += 1

        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if (
            epoch >= MIN_EPOCHS
            and
            patience_counter
            >= EARLY_STOPPING_PATIENCE
        ):

            print(
                "\nEarly stopping."
            )

            break

    total_time = (
        time.time()
        - training_start
    )

    history_df = pd.DataFrame(
        history
    )

    history_path = (
        REPORT_DIR
        / f"{model_name}_history.csv"
    )

    history_df.to_csv(
        history_path,
        index=False
    )

    print("\nTraining finished.")

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        "Best validation macro F1:",
        round(best_macro_f1, 4)
    )

    print(
        "Training time:",
        round(total_time / 60, 2),
        "minutes"
    )

    return {
        "history": history_df,
        "best_epoch": best_epoch,
        "best_macro_f1":
            best_macro_f1,
        "training_seconds":
            total_time
    }